In [8]:
from google.colab import drive
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define a folder in your Drive to store models
cache_dir = '/content/drive/MyDrive/HF_Models_Cache'
os.makedirs(cache_dir, exist_ok=True)

# 3. Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir

print(f"Hugging Face cache set to: {cache_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hugging Face cache set to: /content/drive/MyDrive/HF_Models_Cache


In [9]:
!pip install -U bitsandbytes>=0.46.1

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("SAWithanage/SinLlama-Llama-3-8B-Merged")
model = AutoModelForCausalLM.from_pretrained(
    "SAWithanage/SinLlama-Llama-3-8B-Merged",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [11]:
# 3. Define the Few-Shot Prompt Template
def clean_asr_text(noisy_input):
    # We use strict formatting to force the Base Model into completion mode
    prompt = f"""Noisy: රනගසරට සෞදින් හඩ දැනට කනෙක්ෂණ එක බෙල්පත නිසා දිස්කනක්වලාතිනනේ දැනටව නොවෙඹ කාල සනොවෙබ තිහොදක්වා රුපියල් දහ තුන්දා සාරසිය හැටට දකකෂතැහැට දෙක කොමතල ියෙනවා බිල්පපඊමිලසරහා සෙන්කරන්නේ ිසම්බ විසිදෙක තමයි ඩූඩේටෙක තිබිලා තියෙන නොවෙම්බ බිහටත් පේමට් ික කරත් කනෙක්ෂන් ටික ඔටෝමටික ටික වෙලාවකින් ඇත්ූවෙනවාේන් එකක් රලතියමැඩම් පේමන්ට් ිකක් කරලා තියෙන්නේ දිේබ විසිහට අට දාක්ිට පස්සේ ේමනට් එකක් කරතාමණං අභිටතුලා නෑ පේ කරපු මවන් එක කියද කයල දැනගන්න පුළුවන්ද මැඩ ්මට ඩීටෙල්ස් ටික දෙන්න පොළොන් අනුලින්ද කොොම්ද පේම්ික කරේබැංක කමුට් එකෙන් මැඩම්ගේ මවුන් එක දිඩක් තනාද නකන්ටාන්සික්ෂණත න්සික්ෂණ රිෆරන්ටස් නම් එකත් මැඩම් ලබා ගත්අදඔහුවෝ බෑන්ක් එක් අවුන්ට කේන එක අරිමං එමුනත් කමතෙන එකත් දානමට පේකරපු වෙලාව කියන්නනම්පොඩකින්
Clean: කනෙක්ෂන් එක බිල්පත නිසා ඩිස්කනෙක්ට් වෙලා තියෙන්නේ. දැනට නොවෙම්බර් කාලසීමාව තිහ දක්වා රුපියල් දහතුන්දාස් හාරසිය හැට දෙකක් තිබෙනවා. බිල් පේමන්ට් එක දෙසැම්බර් විසි දෙක තමයි ඩියු ඩේට් එක තිබිලා තියෙන්නේ. නොවෙම්බර් බිලට පේමන්ට් එක කරොත් කනෙක්ෂන් එක ඔටෝමැටිකලි ටික වෙලාවකින් ඇක්ටිව් වෙනවා. මැඩම් පේමන්ට් එකක් කරලා තියෙන්නේ දෙසැම්බර් විසි අටට පස්සේ. පේමන්ට් එකක් කරානම් අපිට ලැබිලා නෑ. පේ කරපු අමවුන්ට් එක කීයද කියලා දැනගන්න පුළුවන්ද? මැඩම් මට ඩීටේල්ස් ටික දෙන්න. කොහොමද පේමන්ට් එක කරේ? බැංකු එකවුන්ට් එකෙන් මැඩම්ගේ අමවුන්ට් එක ට්‍රාන්සැක්ෂන් රිෆරන්ස් නම්බර් එකත් මැඩම් ලබාගන්න. බෑන්ක් එකවුන්ට් එකේ එමුන්ට් එකත් පේ කරපු වෙලාව කියන්නම් පොඩ්ඩක් ඉන්න.

Noisy: ආයිබෝවන් මම රේනුක ට පුළුවනිෝබට සහයවන්න ගේබලත්ම ශකකලබලල කියන්න මැඩ එන්ටකරපු නම් එක අදාලවත කනෙක්ෂණකි වස්කර බිඑකයිකයි විසි හතරයි දහ නම යයි අතසිය හ්ත එකනන් එකද මමඩම් කනේෂ එකට අදාව අයිතිකර්ෙ නම තැනකන්න පුළුවන්තනල මතිමෙරදී ඉන්නාමඩීලකෂ ල බකියන් කරාක ඇමතිමෙ රදීනමට රනදසර සෞදි් නත්දැනනෙක්ෂඑක වෙල්ප නිසා දක්කැන්ලතිය දැනතව ොවෙඹ කාලසනොවෙම්බ තිහ දක්වා රුපියල් දහුන් සාරසය  දෙක්තහැ දකකමදත්තයවා බල්පතීමිලරහසෙනරන්නේ දිසෙම්බ ිසිදෙක තමයි ජඩේක ටිලෙන නොවෙන්බ බහට පීමට් ික කර කනෙක්න් ටික ඔතෝ් තික ටික වෙලාවකින් ත්ූවෙනවා රදපමඩම් ේමන් ෙක් කරලා තියෙන්නේ ඕදිසේබ විසහ අට දිට පත්සයීමන් එිකක්ද තමනං අභ්ෙිලානපේකරපු මෞ්ික කියදකර දැනන්න පුළුවන්ද මට ට ඩීටල්ස් ික දෙන්නුල අනුලින්ද කෝම්ද පීම්ක කරබැකකවුට් එකෙන් මැඩ්ගේ මවුන් එකදඩ්ත ලනන්තාංශික්ෂණ ෙෆරනස් නම් එකත් මැඩම ලබාකත්ත ගේන කම්න කත් දානම ට කරපු වෙලාව කියන්න තයි හතිස් පහයුත්කරපු බැං එක මොකක්දමට නේමේෂන් ත්පැ්එකනහතහතලිස්පහනත වෙලාව මඩ  සංශක්ෂ් රෆ් නම් එක දේන පුළන්ද මට ලෝ මැනතලන්න කෙරෙන්න හනේ ෝඅභ්ට්ටලා නැහැමන මෙතකොට පි කර්නේ බිල්ම්තන ක් දාල කැන්ණ කත්තිය් ක ක බිලින් ප්ින් තමසැලට අ්තනෙත් රමට්ටිය කරන් නං කම්තර එකාන ටික වෙලාවකින් මකික්වයිමටසාංශික්ෂ නම් එකතෙන්න පුළුවන්ද ්්අතසීය හත්තතුනයිදෙසියානු පහතාකන යිතිකරුම ට කතා කරන්නේ මට කනට්මබල් නම් එක මඩ ම කල්කරන නම් කට ම ිල්කම් තැ එකක් ුොර මම යො ොකරපු කක්යන කට දාළ පකරන නම් එක මැඩට කල්කරන මබලි් න් එකට එ සලත් එකක් පන්රකනෙක්ෂ කත් ැත්තයු ක තක්ම ටික වෙලාකන් කකළ බලන්න රි එල්තී මොබිටර මොට සමෙම හරිමඩ වෙනත්ම දැනිරීමට අවශ්‍යද මාලබා දුන් සේවය ඇගම දහ දයු තන්න සතති මොභිටලමොට ස්තුතිය සපත
Clean: ආයුබෝවන් මම රේනුක. පුළුවනි ඔබට සහය වන්න. මැඩම් එන්ටර් කරපු නම්බර් එකට අදාලව කනෙක්ෂන් එක 01124197.. එකද? මැඩම් කනෙක්ෂන් එකට අදාල අයිතිකරුගේ නම දැනගන්න පුළුවන්ද? අමතිරීදි ඉන්නා මධුලක්ෂිකා. කරුණාකර ඇමතුමේ රැඳී සිටින්න. දැනට කනෙක්ෂන් එක බිල්පත නිසා ඩිස්කනෙක්ට් වෙලා තියෙන්නේ. දැනට නොවෙම්බර් තිහ දක්වා රුපියල් දහතුන්දාස් හාරසිය හැත්තෑ දෙකක් තිබෙනවා. බිල්පත දෙසැම්බර් විසි දෙක තමයි ඩියු ඩේට් එක. නොවෙම්බර් බිලට පේමන්ට් එක කරොත් කනෙක්ෂන් ටික ඔටෝමැටිකලි ටික වෙලාවකින් ඇක්ටිව් වෙනවා. මැඩම් පේමන්ට් එකක් කරලා තියෙන්නේ දෙසැම්බර් විසි අටට පස්සේ. පේමන්ට් එකක් කරානම් අපිට ලැබිලා නෑ. පේ කරපු අමවුන්ට් එක කීයද කියලා දැනගන්න පුළුවන්ද? මට ඩීටේල්ස් ටික දෙන්න. කොහොමද පේමන්ට් එක කරේ? බැංකු එකවුන්ට් එකෙන් මැඩම්ගේ අමවුන්ට් එක ට්‍රාන්සැක්ෂන් රිෆරන්ස් නම්බර් එකත් මැඩම් ලබාගන්න. පේ කරපු වෙලාව කියන්න. හතළිස් පහයි... කරපු බැංකුව මොකක්ද? ඉන්ෆොර්මේෂන් එක දෙන්න පුළුවන්ද? ට්‍රාන්සැක්ෂන් රිෆරන්ස් නම්බර් එක දෙන්න පුළුවන්ද? කනෙක්ෂන් අයිතිකරුමද කතා කරන්නේ? මට කන්ටැක්ට් නම්බර් එක මැඩම් කෝල් කරන නම්බර් එකට බිලින් එකට මැසේජ් එකක් එවන්නම්. කනෙක්ෂන් එක ටික වෙලාවකින් ඇක්ටිව් වෙයි. වෙනත් දැනගැනීමට අවශ්‍යද? මා ලබා දුන් සේවය ඇගයීමට රැඳී සිටින්න. ස්තූතියි.

Noisy: එබවන් මම රක්ෂිමට පුුව්න බිඩ සහයවන්නලෝසගේියන්න්ලෑස්ටු නදිකය්කිපැකජ් එකකයිදව ට දනග ලුතන් දනනේේන්නමිටෝම් ප්ලස්සියලවසදග ගෙන කොඩක් වසර ෙනපුළ්අනමිටඩ් හෝම් ්ලස් පැකේජ්කේ රේැඳනසාරඳසනට ස්තූතියි අිිටඩ් හෝම් ප්ලස්පැකේජේ රෙන්ටලික ිපල් නවදාස් නවසේයයි සර ඒකස්පී්එක ටුහන් රෙන්බිපිය්ටිනවා දවනලෝඩ්ස්පීඩ් එක ව්හන්රන්බියස් ටියනවා අප්ලෝඩ්්ීඩ් එක රිස්ත්‍රික්ෂණඑකක් නැහැ අනුලමිට ඩියුස් කරන්න පුළුව් පොයිස් කෝල්සු ත්ලිමිටත් ගන්න ුළුවන්සාතකොටන්නමිට් හෝ් කිය එයිමේ ලස්සගෙි ෙනස්ක ්පීඩ් එක මගේ දැන් ්්ක්ටු ෙන්ි පයිගෙන අලුත පැකේකින් ැද්ෆ්ලෑස්න්ටිෆර්කත්තයන ආාන දන් ම අන්ිමට හම ්ල්සගෙන මාරුන්  ආයවනම් ෆ්ලෑ්ු වෙඩ පරගන මාරෙන් පුුවන් පුළුවන්සම කොහොමද න්ලිට් හම ප්ලස්සකෙනමාවෙන්මෙතනින්සට මාරු කරල දන්න පුළුවන් කනක්ෂණ් එක යිටිකරගේ අඩී න්ෙක්කියන්න ුුවන්ද සටයයිහැටහ්තයහේටඋනිහනේ  අසු නහසකරනක ඇමතමේ ර්දෙන්නමතිම රඳින් සිටට ස්තූතය සහද දවසත්ුද පකජ්එකබ්ඩේට් වනවන්සට තකල් ගේ ද්ටමේ තියන පැකේකේ හුස් කරන්න පුළුවන් කොහොමත් මෙපකේ්කට දින ෙක්කට දාලවලන්තක ඇඩ්ද ාමේ මාසේ බිල් එකට හෙුවෙ මෙමනසේ සම අද දවසදී මේ පැකජකේ් රන්න දවඇතලත ලුත් පැකේජ් ක අප්ඩට ෙන සාගැන මට ය පස්සේ ඕනොත් මද අඩු ැින් එකරු් මාරුවෙන්න පුළුවන් පුළුවන්සගැටක් නැහැ ගනට කාලයක් තයනා කජ්යක් දාල ්පට ා් ඉන්නඕනනිවාරයන්එහෙම එකක්න ැබැයි මේඩෞන්ග්‍රේඩ් එකක් ක්‍රික්වස් කරනවානංසව දෞ්්‍කැනිදන්ම යාලිමිට ෆයබ ්ලෑන්ලතයක් ිහෙමමකුත් වරට්රස් එකක් නැහැ ඒත් නොමකෙනඩෞන්ග්‍රේඩ් එකක් යනවනම් බිලින් ිකටතමයි න්ග‍රේට් කරගන් ළගෙන්ගෙන න මසකාලවෙනිඩ විතරයසලෙසතමබි්තුතේ සැප දවසක්ස
Clean: ආයුබෝවන් මම රක්ෂිත. පුළුවනි ඔබට සහය වන්න. අන්ලිමිටඩ් හෝම් ප්ලස් පැකේජ් එක ගැන දැනගන්න පුළුවන්. අන්ලිමිටඩ් හෝම් ප්ලස් පැකේජ් එකේ රෙන්ටල් එක රුපියල් නවදහස් නවසියයයි. ඩවුන්ලෝඩ් ස්පීඩ් එක 100 Mbps තියෙනවා. අප්ලෝඩ් ස්පීඩ් එක රිස්ට්‍රික්ෂන් එකක් නැහැ. අන්ලිමිටඩ් යූස් කරන්න පුළුවන්. වොයිස් කෝල්ස් පවා අන්ලිමිටඩ් ගන්න පුළුවන්. පැකේජ් එක මාරු කරගන්න පුළුවන්. මාරු කරන්න පුළුවන් කොහොමද අන්ලිමිටඩ් හෝම් ප්ලස් එකට? මෙතනින් මාරු කරලා දෙන්න පුළුවන්. කනෙක්ෂන් එකේ අයිතිකරුගේ අයිඩී නම්බර් එක කියන්න පුළුවන්ද? 8776... කරුණාකර ඇමතුමේ රැඳී සිටින්න. ස්තූතියි සහෘද දවසක්. පැකේජ් එක අප්ඩේට් වෙනකල් දැනට තියෙන පැකේජ් එක යූස් කරන්න පුළුවන්. මේ මාසේ බිල් එකට ඇඩ් වෙයි. අද දවසේදී මේ පැකේජ් එක ඇක්ටිව් වෙයි. අලුත් පැකේජ් එක අප්ඩේට් වුනාට පස්සේ ඕනෙ නම් අඩු පැකේජ් එකකට මාරු වෙන්න පුළුවන්. ගැටලුවක් නැහැ. යම් කාලයක් තියෙන්න ඕනේ කියලා එකක් නැහැ. හැබැයි ඩවුන්ග්‍රේඩ් එකක් රික්වෙස්ට් කරනවා නම් බිලින් එකට තමයි ඩවුන්ග්‍රේඩ් කරගන්න ඕනේ. ස්තූතියි, සුබ දවසක්.

Noisy: යිබෝවන් මරානිමඩපොලනිපුට සහය වන්නයඩුව ේමිස් අපේ පියෝටීවි එකයි තෙලිපුරනයින් එකයි දෙකම වැඩ නැහැටමීට දවස් දපලින් විල්ලත් හැුවා හැදුවත ඒ විදිහටම ආය මේ නැව  ක්තිය වෙලා තිෙනවාතෙලිෝන් එයි පියෝටීවි එකයිකනක්ෂණ එකිටකරුගෙනමෙන්ඩබ්ලි පීපී ඒ කුමාරපයිබලයින් එකක්යලින්ත්‍රයඩ් එක වැඩකරනදනෑමොකම ැඩනෑඑල්ලවසි ගෙන ලයි් කරෙඩ්වලා තින්ද බලනවෙන් වෙලා ියරෙඩ්වෙලා තයෙනවම්පෙන් එක ඇතුළත් කරන්නෑ මට සම්බලගරන් මභය නම්ම ් බින්ද හතයි හයයිරිඅසුතුනයි හඅනු එ්කයි රිටසිය පනස්පහැතිමි බැිනොනව ඳිනස නගත ස්තුතියි ම අදාලඩපන්හදනටීමකට එක ව ක්පෙන් එක දන්නෙල ලාවක්පනබය නම්බකටසෙර්ත් එකක් ගවල් මසරීමඩල ති්තූය සබ දවසසක් මෙව මැගම සඳා රැඳී සිටින්න
Clean: ආයුබෝවන් මම මරානි. පුළුවනි ඔබට සහය වන්න. අපේ පියෝ ටීවී එකයි ටෙලිෆෝන් ලයින් එකයි දෙකම වැඩ නැහැ. මීට දවස් දෙකකට කලින් හැදුවා. හැදුවත් ඒ විදිහටම ආයෙත් මේක වෙලා තියෙනවා. කනෙක්ෂන් අයිතිකරු කුමාර. මොකුත් වැඩ නැහැ. රතු ලයිට් එක පත්තු වෙලා තියෙනවා. මට සම්බන්ධ කරගන්න පුළුවන් මොබයිල් නම්බර් එක 076... ස්තූතියි. අදාල අංශයට දැනුම් දීමක් කරන්නම්. මැසේජ් එකක් එයි. රැඳී සිටින්න. ස්තූතියි, සුබ දවසක්.

Noisy: ඇල්බැන්මන් කයලූ මටු පුළුවන්න සාය න්න ආය්බැ්මනනවෝ මටපුළුව් උබසහය න මිස්න්ඩීනිකැත පාප්ෝන් එක වැඩකරන්න් නෑ ඉතනෙඩ කර්ේන ංග්රේෂ්ෝන එකක් වගේ වෙන්නේ ි්තන පැඩදුරදුරකථන අන් ේකයන්නමැඩ හඩු ඇතිස් කතයිූ දෙකයිටිඔවූ දෙසව ක ඉන්නෙට් එක් ඒෙම තමයි යන්නේ දෙක වැඩ කර්නේමේක පයිප කනෙක්ෂියන් එකක් තියෙන්න මැඩම්ගේ නෞ්ප විකිය බල්නා පීඕ එන් කියලායිෙක කොහොමද පත්තුවෙන්න කියලා පී අගුරය මවුරා එන්නකුරය රොග් කේපීපී ඕ එන් කියන එකය එන් පත්තුවෙන්නල්ලෝ එපප ෝඑන් කෙරලා යිෙක පත්තවන් නතුනත් කනක්ෂ මොකුත් වැඩකිරීම කරන් නැහැෑ කරනා කරනන්දසිටිමඩ ේ කතාගන නම්බරඑකෙන් සම්බන්ධ කිරීම කරගන්න පුළුවන්ඳු මැඩ ව්පුලිඇම්තනදෙසටන්නම්තනදී සිටින මැඩම් තවදරටා ම්තනදි සිටියාට ස්තූතියි මම ෙතනීන් දනු්දීම කරා මැඩ මදාළ අංශයට පැමිණ්ලි ාේය ්ෙම් මස් එකක් එනවා  දැනවීමක් රා මැමෙනත්‍යමත් දනගැනීමට අවශ්‍ය ද ම කරන්නෑ කම්රේන් කරාල දැනුම් දීමක් කරා මැඩම් පැමිණිලි අංක මැඩම්ටරස්ස් මෙස් එකක් එනව මම ෙතනීන් දැනු්ම් කර මඩ ස්වල්ට මොබිටල් මුවාදි ස්තතියි සුබදවකක හමා ලපබාතුන් සේවයගේ සඳහරනදසට
Clean: ආයුබෝවන් මම කැලුම්. මට පුළුවන් ඔබට සහය වන්න. ෆෝන් එක වැඩ කරන්නේ නෑ. ඉන්ටර්නෙට් කනෙක්ෂන් එකත් වැඩ කරන්නේ නෑ. දුරකථන අංකය කියන්න මැඩම්. මැඩම්ගේ රවුටර් එකේ PON කියලා එක කොහොමද පත්තු වෙන්නේ? PON පත්තු වෙන්නේ නැත්නම් කනෙක්ෂන් මොකුත් වැඩ කරන්නේ නැහැ. මැඩම් මේ කතා කරන නම්බර් එකෙන් සම්බන්ධ කරගන්න පුළුවන්ද? මැඩම් තවදුරටත් රැඳී සිටියාට ස්තූතියි. මම මෙතනින් දැනුම් දීමක් කරා මැඩම්. අදාල අංශයට පැමිණිල්ලක් දාලා මැසේජ් එකක් එයි. වෙනත් යමක් දැනගැනීමට අවශ්‍යද? SLT මොබිටෙල් සමඟ රැඳී සිටියාට ස්තූතියි. සුබ දවසක්.

Noisy: {noisy_input}
Clean:"""

    # Tokenize the prompt and send to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 4. Generate Output with Strict Constraints
    outputs = model.generate(
        **inputs,
        max_new_tokens=512, # Limit generation length
        temperature=0.1,    # Keep it highly deterministic to avoid hallucinations
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode the raw output from the model
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 5. Apply Manual Stopping Criteria
    # Slice off the original prompt text so we only have the new generation
    generated_text = full_output[len(prompt):].strip()

    # Base models might try to keep writing (e.g., generating a new "Noisy:" line).
    # We force it to stop by splitting at the first newline character.
    clean_output = generated_text.split('\n')[0].strip()

    return clean_output

In [12]:
# --- Test the Implementation ---
print("\n--- System Ready ---")
test_noisy_text = "හෙලෝ ආයුබෝවන් මද සම්බන්ය පුළුවනි ඔබට සහාය වන්න ආයුබෝවන් මිසරි ටීවී ඒකේ තියෙනවේ රවුටල් එක වැඩ කරන්නේ නැතු වෙන්නේ නැති මේ ටීවී ඒක වැඩ කරන්නේ න අටීවී අර රවුට සුභ පාට් එක නෑ නම්බර් එක කියන්න ංදුවයි හයි හතයි හරි හ් යි යි යි යි යි ය නම තියෙන්ේනේ කොහොමද හිමිකරුගේ කතිමුණසිවා හරි ඒ කියන්නේ බල් ගොඩක් තියෙන එකේද කලුපාට පොඩි බොක්ස් එකේද සර් කියන්න නෑ කර් බල්බ් ගොඩක් තියෙන එකේ ඒක මේ දැන් කාලන්නට් එක ආව මේක වැඩ කරන්න ඕනේ නේ බල්බ් වැඩම් කරන්නේ නෑහැ ඒක බල්බඅර රවන් වෙන්නේ නැද්ද එකත්පත්තු වෙලා ඒක ඩිංගක ින්නේ වෙනවා අර කල පාට මේ කරන්ට් එක පෙන්නා නෑ ටෙලිෆෝන් එකේ ප්රශ්නයක් නෑැ නේද මොනවත්ම නෑ දැනිෆෝන් එකේ ප්රශ්නයක්නේ නෑ ආ හරි ටෙලිෆෝන් ක් ක් බලන්න ඩඉන්නම් මේමේ ඉන්ටර්නේ ෝ ටෙලිෆෝන් එකේ ප්රස් නයක්නේ නැහැේ දේ ඉන්ටර්නෙට් වෙනදාට යූස් කරනවා නේද ඔව් ඔව් හරි මිස් ඒ රවුටර්ේ නැතුව ටීවී බලන්නේ බෑ නේ බෑ සර් ඉන්ටර්නෙට් පියෝ ටීවී දෙකටම රවුටර් එක ඔව් මේ හරියටම වැඩකරනවා නම් තමයි ක යූ ටීවූ අරමේ මොකුත්ත එන්නේ නෑ ඒ නිසා නේ ඔව් සර් කන්ටැක්ට් කරගන්න මොබයිල් නම්බර් එකත් කියන්න බංදවයි හඒවායි හරි න්යි ඔව් හතයි ඒකයි යි හොඳයි සර් කම්ප්ලේන් එක යොමු කරා ටෙක්නිකල් අංශයෙන් කෝල් එක් දෙයි සර්ට හොඳයිල හරි වෙනත් යමක් දැනගන්න අවශ්යද නෑැ මේ මේ මේ මොක්දැන් මේ රවුටර් එක මේහේ ඔෆිස් එකට අරන් ගියා වැඩක් වෙන්නේ නෑනේ මිස් නෑ ඒකනේ හෙම අවශ්ය නැහැ එගොල්ල චෙක් කරලා එහෙම ගැටලුවක් තියෙනවා නම් මාරු කරලා දේයි සර් හරි මිස හරි හරි හොඳයි මා ලබා දුන් සේවයගේම සඳහා රැඳී සිටින්න එස්එල්ටී මොබිටේල් ඇමතුවට ස්තූතියි සුභ දවසක් බෝවන් බවන්මේ ස්තූතියියනි හරි හරි"

print(f"INPUT: {test_noisy_text}")
cleaned = clean_asr_text(test_noisy_text)
print(f"CLEANED: {cleaned}")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- System Ready ---
INPUT: හෙලෝ ආයුබෝවන් මද සම්බන්ය පුළුවනි ඔබට සහාය වන්න ආයුබෝවන් මිසරි ටීවී ඒකේ තියෙනවේ රවුටල් එක වැඩ කරන්නේ නැතු වෙන්නේ නැති මේ ටීවී ඒක වැඩ කරන්නේ න අටීවී අර රවුට සුභ පාට් එක නෑ නම්බර් එක කියන්න ංදුවයි හයි හතයි හරි හ් යි යි යි යි යි ය නම තියෙන්ේනේ කොහොමද හිමිකරුගේ කතිමුණසිවා හරි ඒ කියන්නේ බල් ගොඩක් තියෙන එකේද කලුපාට පොඩි බොක්ස් එකේද සර් කියන්න නෑ කර් බල්බ් ගොඩක් තියෙන එකේ ඒක මේ දැන් කාලන්නට් එක ආව මේක වැඩ කරන්න ඕනේ නේ බල්බ් වැඩම් කරන්නේ නෑහැ ඒක බල්බඅර රවන් වෙන්නේ නැද්ද එකත්පත්තු වෙලා ඒක ඩිංගක ින්නේ වෙනවා අර කල පාට මේ කරන්ට් එක පෙන්නා නෑ ටෙලිෆෝන් එකේ ප්රශ්නයක් නෑැ නේද මොනවත්ම නෑ දැනිෆෝන් එකේ ප්රශ්නයක්නේ නෑ ආ හරි ටෙලිෆෝන් ක් ක් බලන්න ඩඉන්නම් මේමේ ඉන්ටර්නේ ෝ ටෙලිෆෝන් එකේ ප්රස් නයක්නේ නැහැේ දේ ඉන්ටර්නෙට් වෙනදාට යූස් කරනවා නේද ඔව් ඔව් හරි මිස් ඒ රවුටර්ේ නැතුව ටීවී බලන්නේ බෑ නේ බෑ සර් ඉන්ටර්නෙට් පියෝ ටීවී දෙකටම රවුටර් එක ඔව් මේ හරියටම වැඩකරනවා නම් තමයි ක යූ ටීවූ අරමේ මොකුත්ත එන්නේ නෑ ඒ නිසා නේ ඔව් සර් කන්ටැක්ට් කරගන්න මොබයිල් නම්බර් එකත් කියන්න බංදවයි හඒවායි හරි න්යි ඔව්